# SVD-Based Book Recommender System - Prototype Comparison

This notebook compares:
- **Individual recommendations** using SVD matrix factorization
- **Group recommendations** using Average and Least Misery strategies
- **Baseline recommenders** (Popularity, Average Rating, Random)

## Status Legend
- ✅ **IMPLEMENTED**: Fully functional
- 🔄 **IN PROGRESS**: Core functionality working, tuning needed
- 📋 **PLANNED**: Design complete, not yet implemented

## Setup

In [6]:
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')

from recommenders import (
    SVDRecommender,
    GroupRecommender,
    PopularityBaseline,
    AverageRatingBaseline,
    train_test_split_temporal
)
from metrics import precision_at_k, recall_at_k

print("✅ Imports successful")

✅ Imports successful


In [7]:
# Load data
print("Loading data...")
ratings_df = pd.read_csv('../data/ratings.csv')
books_df = pd.read_csv('../data/books.csv')

print(f"Ratings: {len(ratings_df):,} | Books: {len(books_df):,} | Users: {ratings_df['user_id'].nunique():,}")
print(f"Sparsity: {(1 - len(ratings_df)/(ratings_df['user_id'].nunique() * len(books_df)))*100:.2f}%")

Loading data...
Ratings: 5,976,479 | Books: 10,000 | Users: 53,424
Sparsity: 98.88%


## ✅ Phase 1: Train/Validation/Test Split

In [8]:
# Sample for faster demo (use full data for production)
sample_ratings = ratings_df.sample(frac=0.15, random_state=42)

train, val, test = train_test_split_temporal(
    sample_ratings,
    train_ratio=0.6,
    val_ratio=0.2
)

print(f"✅ Train: {len(train):,} ratings (60%)")
print(f"✅ Validation: {len(val):,} ratings (20%)")
print(f"✅ Test: {len(test):,} ratings (20%)")

✅ Train: 537,883 ratings (60%)
✅ Validation: 179,294 ratings (20%)
✅ Test: 179,295 ratings (20%)


## ✅ Phase 2: Train SVD Model

In [9]:
print("Training SVD model with 50 factors...")
svd_model = SVDRecommender(n_factors=50, random_state=42)
svd_model.fit(train)

print(f"✅ Model trained")
print(f"   Users: {len(svd_model.user_idx_map):,}")
print(f"   Items: {len(svd_model.item_idx_map):,}")
print(f"   Global mean: {svd_model.global_mean:.2f}")

Training SVD model with 50 factors...
✅ Model trained
   Users: 53,375
   Items: 9,997
   Global mean: 3.92


## ✅ Individual Recommendations Example

In [10]:
# Select a test user
user_rating_counts = train.groupby('user_id').size().sort_values(ascending=False)
test_user_id = user_rating_counts.index[10]

user_history = train[train['user_id'] == test_user_id].merge(
    books_df[['book_id', 'title', 'authors', 'average_rating']],
    on='book_id'
)

print(f"\n{'='*80}")
print(f"INDIVIDUAL RECOMMENDATIONS - User {test_user_id}")
print(f"{'='*80}")
print(f"\nUser's Rating History ({len(user_history)} books):")
for i, (_, row) in enumerate(user_history.head(5).iterrows(), 1):
    print(f"  {i}. {row['title'][:50]:50s} | Rating: {row['rating']}")
if len(user_history) > 5:
    print(f"  ... and {len(user_history) - 5} more")


INDIVIDUAL RECOMMENDATIONS - User 23894

User's Rating History (26 books):
  1. The Rape of Nanking                                | Rating: 3
  2. The Complete Tales and Poems of Winnie-the-Pooh (W | Rating: 4
  3. Clifford the Big Red Dog                           | Rating: 4
  4. Uncle Tom's Cabin                                  | Rating: 5
  5. The Little House Collection (Little House, #1-9)   | Rating: 5
  ... and 21 more


In [11]:
# Get recommendations
exclude_books = set(user_history['book_id'].values)
recommendations = svd_model.recommend(
    test_user_id,
    n_recommendations=10,
    exclude_items=exclude_books
)

rec_books = books_df[books_df['book_id'].isin(recommendations.index)].merge(
    pd.DataFrame({
        'book_id': recommendations.index,
        'predicted_rating': recommendations.values
    }),
    on='book_id'
).sort_values('predicted_rating', ascending=False)

print(f"\n✅ Top 10 SVD Recommendations for User {test_user_id}:")
for i, (_, row) in enumerate(rec_books.head(10).iterrows(), 1):
    print(f"  {i:2d}. {row['title'][:50]:50s} | Pred: {row['predicted_rating']:.2f}")


✅ Top 10 SVD Recommendations for User 23894:
   1. The Calvin and Hobbes Tenth Anniversary Book       | Pred: 5.00
   2. The Complete Calvin and Hobbes                     | Pred: 5.00
   3. It's a Magical World: A Calvin and Hobbes Collecti | Pred: 5.00
   4. The Calvin and Hobbes Lazy Sunday Book             | Pred: 5.00
   5. There's Treasure Everywhere: A Calvin and Hobbes C | Pred: 5.00
   6. The Declaration of Independence and The Constituti | Pred: 5.00
   7. ESV Study Bible                                    | Pred: 5.00
   8. The Absolute Sandman, Volume One                   | Pred: 5.00
   9. The Sweet Life in Paris: Delicious Adventures in t | Pred: 5.00
  10. Cinderella                                         | Pred: 5.00


## ✅ Group Recommendations Comparison

In [12]:
# Create a group of 4 users
group_user_ids = user_rating_counts.index[5:9].tolist()

print(f"\n{'='*80}")
print(f"GROUP RECOMMENDATIONS - Group of 4 Users")
print(f"{'='*80}")
print(f"\nGroup members: {group_user_ids}")
for uid in group_user_ids:
    n_ratings = len(train[train['user_id'] == uid])
    print(f"  User {uid}: {n_ratings} ratings")

# Exclude books rated by any group member
group_exclude = set(train[train['user_id'].isin(group_user_ids)]['book_id'].values)


GROUP RECOMMENDATIONS - Group of 4 Users

Group members: [20076, 23612, 11268, 21987]
  User 20076: 27 ratings
  User 23612: 27 ratings
  User 11268: 27 ratings
  User 21987: 27 ratings


In [13]:
# Generate recommendations with different strategies
group_recommender = GroupRecommender(svd_model)

print(f"\n--- AVERAGE Strategy (maximize group satisfaction) ---")
avg_recs = group_recommender.average_strategy(
    group_user_ids,
    n_recommendations=10,
    exclude_items=group_exclude
)

avg_books = books_df[books_df['book_id'].isin(avg_recs.index)].merge(
    pd.DataFrame({
        'book_id': avg_recs.index,
        'avg_rating': avg_recs.values
    }),
    on='book_id'
).sort_values('avg_rating', ascending=False)

print("Top 5:")
for i, (_, row) in enumerate(avg_books.head(5).iterrows(), 1):
    print(f"  {i}. {row['title'][:50]:50s} | Avg Pred: {row['avg_rating']:.2f}")


--- AVERAGE Strategy (maximize group satisfaction) ---
Top 5:
  1. The Holy Bible: English Standard Version           | Avg Pred: 4.64
  2. The Complete Tales and Poems                       | Avg Pred: 4.64
  3. The Sweet Life in Paris: Delicious Adventures in t | Avg Pred: 4.64
  4. Twenty Wishes (Blossom Street, #5)                 | Avg Pred: 4.64
  5. The Divan                                          | Avg Pred: 4.64


In [14]:
print(f"\n--- LEAST MISERY Strategy (ensure no one unhappy) ---")
lm_recs = group_recommender.least_misery_strategy(
    group_user_ids,
    n_recommendations=10,
    exclude_items=group_exclude
)

lm_books = books_df[books_df['book_id'].isin(lm_recs.index)].merge(
    pd.DataFrame({
        'book_id': lm_recs.index,
        'min_rating': lm_recs.values
    }),
    on='book_id'
).sort_values('min_rating', ascending=False)

print("Top 5:")
for i, (_, row) in enumerate(lm_books.head(5).iterrows(), 1):
    print(f"  {i}. {row['title'][:50]:50s} | Min Pred: {row['min_rating']:.2f}")


--- LEAST MISERY Strategy (ensure no one unhappy) ---
Top 5:
  1. Hafalan Shalat Delisa                              | Min Pred: 4.27
  2. The Sweet Life in Paris: Delicious Adventures in t | Min Pred: 4.26
  3. The Divan                                          | Min Pred: 4.26
  4. EntreLeadership: 20 Years of Practical Business Wi | Min Pred: 4.26
  5. Twenty Wishes (Blossom Street, #5)                 | Min Pred: 4.26


In [15]:
# Compare strategies
avg_set = set(avg_recs.index)
lm_set = set(lm_recs.index)
overlap = avg_set & lm_set

print(f"\n✅ Strategy Comparison:")
print(f"   Overlap: {len(overlap)}/10 items ({len(overlap)/10*100:.0f}%)")
print(f"   Unique to Average: {len(avg_set - lm_set)}")
print(f"   Unique to Least Misery: {len(lm_set - avg_set)}")
print(f"\n💡 Insight:")
print(f"   Average strategy optimizes for group happiness")
print(f"   Least Misery ensures no member is left behind")


✅ Strategy Comparison:
   Overlap: 10/10 items (100%)
   Unique to Average: 0
   Unique to Least Misery: 0

💡 Insight:
   Average strategy optimizes for group happiness
   Least Misery ensures no member is left behind


## ✅ Baseline Comparison

In [16]:
print(f"\n{'='*80}")
print(f"BASELINE COMPARISON")
print(f"{'='*80}")

pop_baseline = PopularityBaseline(books_df)
rating_baseline = AverageRatingBaseline(books_df)

# Get recommendations from each method
svd_recs = set(svd_model.recommend(test_user_id, n_recommendations=10, exclude_items=exclude_books).index)
pop_recs = set(pop_baseline.recommend(n_recommendations=10, exclude_items=exclude_books))
rating_recs = set(rating_baseline.recommend(n_recommendations=10, exclude_items=exclude_books))

print(f"\n✅ SVD Model: {len(svd_recs)} recommendations")
print(f"✅ Popularity Baseline: {len(pop_recs)} recommendations")
print(f"✅ Average Rating Baseline: {len(rating_recs)} recommendations")

print(f"\nOverlap Analysis:")
print(f"  SVD vs Popularity: {len(svd_recs & pop_recs)}/10 items")
print(f"  SVD vs Rating: {len(svd_recs & rating_recs)}/10 items")
print(f"  Popularity vs Rating: {len(pop_recs & rating_recs)}/10 items")


BASELINE COMPARISON

✅ SVD Model: 10 recommendations
✅ Popularity Baseline: 10 recommendations
✅ Average Rating Baseline: 10 recommendations

Overlap Analysis:
  SVD vs Popularity: 0/10 items
  SVD vs Rating: 4/10 items
  Popularity vs Rating: 0/10 items


In [17]:
print(f"\n--- Popularity Baseline Top 5 ---")
pop_books = books_df[books_df['book_id'].isin(pop_recs)].nlargest(5, 'ratings_count')
for i, (_, row) in enumerate(pop_books.iterrows(), 1):
    print(f"  {i}. {row['title'][:50]:50s} | Ratings: {row['ratings_count']:,}")

print(f"\n--- Average Rating Baseline Top 5 ---")
rating_books = books_df[books_df['book_id'].isin(rating_recs)].nlargest(5, 'average_rating')
for i, (_, row) in enumerate(rating_books.iterrows(), 1):
    print(f"  {i}. {row['title'][:50]:50s} | Avg Rating: {row['average_rating']:.2f}")


--- Popularity Baseline Top 5 ---
  1. The Hunger Games (The Hunger Games, #1)            | Ratings: 4,780,653
  2. Harry Potter and the Sorcerer's Stone (Harry Potte | Ratings: 4,602,479
  3. Twilight (Twilight, #1)                            | Ratings: 3,866,839
  4. The Great Gatsby                                   | Ratings: 2,683,664
  5. The Fault in Our Stars                             | Ratings: 2,346,404

--- Average Rating Baseline Top 5 ---
  1. The Complete Calvin and Hobbes                     | Avg Rating: 4.82
  2. Words of Radiance (The Stormlight Archive, #2)     | Avg Rating: 4.77
  3. Harry Potter Boxed Set, Books 1-5 (Harry Potter, # | Avg Rating: 4.77
  4. ESV Study Bible                                    | Avg Rating: 4.76
  5. Mark of the Lion Trilogy                           | Avg Rating: 4.76


## 🔄 Validation & Hyperparameter Tuning

In [18]:
# Evaluate on test set
print(f"\n{'='*80}")
print(f"VALIDATION METRICS (Test Set)")
print(f"{'='*80}")

test_sample = test.sample(min(100, len(test)), random_state=42)
predictions = []
actuals = []

for _, row in test_sample.iterrows():
    try:
        pred = svd_model.predict(row['user_id'], [row['book_id']])[row['book_id']]
        predictions.append(pred)
        actuals.append(row['rating'])
    except:
        pass

predictions = np.array(predictions)
actuals = np.array(actuals)

rmse = np.sqrt(np.mean((predictions - actuals) ** 2))
mae = np.mean(np.abs(predictions - actuals))

print(f"\n✅ Test Set Evaluation ({len(actuals)} predictions):")
print(f"   RMSE: {rmse:.3f}")
print(f"   MAE:  {mae:.3f}")
print(f"   Range: [1.0, 5.0]")
print(f"\n💡 Interpretation: Predictions are off by ~{mae:.2f} rating points on average")


VALIDATION METRICS (Test Set)

✅ Test Set Evaluation (100 predictions):
   RMSE: 1.055
   MAE:  0.823
   Range: [1.0, 5.0]

💡 Interpretation: Predictions are off by ~0.82 rating points on average


In [19]:
# 🔄 Hyperparameter tuning (in progress)
print(f"\n🔄 HYPERPARAMETER TUNING (In Progress)")
print(f"\nCurrent setting: n_factors=50")
print(f"Planned search: n_factors in [10, 25, 50, 75, 100]")
print(f"Evaluation metric: Validation set RMSE")
print(f"Status: Early stopping framework ready")


🔄 HYPERPARAMETER TUNING (In Progress)

Current setting: n_factors=50
Planned search: n_factors in [10, 25, 50, 75, 100]
Evaluation metric: Validation set RMSE
Status: Early stopping framework ready
